[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/08_numerical_ode_solvers/first_principles.ipynb)

# Topic 08: Numerical ODE Solvers

## 1. First-Principles Intuition & Motivation

An initial value problem states the law of motion and the starting point:

$$
\mathbf{y}'(t) = \mathbf{f}(t, \mathbf{y}(t)), \qquad \mathbf{y}(t_0) = \mathbf{y}_0, \qquad \mathbf{y} : [t_0, T] \to \mathbb{R}^{d} .
$$

Picard–Lindelöf says that if $\mathbf{f}$ is continuous in $t$ and Lipschitz in $\mathbf{y}$, a unique solution exists on some interval — see [`../../differential_equations/02_existence_uniqueness_picard_lindelof/`](../../differential_equations/02_existence_uniqueness_picard_lindelof/) for that theory, which we take as given here. What existence does *not* give is a formula. Outside a small catalogue of separable, linear, and exact equations, no closed form exists, and for nonlinear systems in more than two dimensions the trajectories can be chaotic. Numerical integration is the only general tool.

**The one identity behind every method.** Integrate the ODE over one step:

$$
\mathbf{y}(t_{n+1}) = \mathbf{y}(t_n) + \int_{t_n}^{t_n + h} \mathbf{f}(s, \mathbf{y}(s))\,ds .
$$

This is exact. The difficulty is that the integrand depends on the unknown $\mathbf{y}$, so we cannot evaluate it. **Every numerical method is a quadrature rule for this integral, using approximations of $\mathbf{y}$ that the method itself generates.**

- Left-endpoint rectangle $\Rightarrow$ $\mathbf{y}_{n+1} = \mathbf{y}_n + h\mathbf{f}(t_n, \mathbf{y}_n)$: **forward Euler**, explicit.
- Right-endpoint rectangle $\Rightarrow$ $\mathbf{y}_{n+1} = \mathbf{y}_n + h\mathbf{f}(t_{n+1}, \mathbf{y}_{n+1})$: **backward Euler**, implicit (the unknown appears on both sides).
- Trapezoid $\Rightarrow$ $\mathbf{y}_{n+1} = \mathbf{y}_n + \tfrac{h}{2}[\mathbf{f}(t_n,\mathbf{y}_n) + \mathbf{f}(t_{n+1},\mathbf{y}_{n+1})]$: **trapezoidal / Crank–Nicolson**, implicit, order 2.
- Interior sample points with predicted values $\Rightarrow$ **Runge–Kutta**.
- Polynomial interpolation through *past* $\mathbf{f}$ values $\Rightarrow$ **Adams multistep** methods.

That reduction is why Topic 06 on quadrature is the direct prerequisite: the accuracy of the quadrature rule is where the order of the ODE method comes from.

### The two things that can go wrong

**Accuracy.** One step of forward Euler is a truncated Taylor expansion, so it is wrong by $O(h^2)$. Over $T/h$ steps those errors accumulate. If they merely add, the total is $O(h^2)\cdot O(1/h) = O(h)$ — one order is *always* lost between local and global error. But they do not merely add: each error is then propagated by the dynamics itself, and if the dynamics is expanding, errors grow. Controlling that amplification is what the Gronwall lemma does, and its conclusion (global error $\le \frac{Ch}{L}(e^{L(T-t_0)} - 1)$) is honest about the exponential price of long integrations of unstable systems.

**Stability.** This is the subtler failure, and it is not about accuracy at all. Consider $y' = -1000y$, $y(0) = 1$: the solution collapses to zero almost instantly, and any sensible answer after $t = 0.01$ is "essentially zero". Forward Euler gives $y_{n+1} = (1 - 1000h)y_n$, so unless $h \lt 2/1000$ the factor $\vert 1 - 1000h\vert$ exceeds $1$ and the *numerical* solution oscillates with exponentially growing amplitude — a catastrophic answer to a trivially decaying problem. Backward Euler gives $y_{n+1} = y_n/(1 + 1000h)$, which decays for **every** $h \gt 0$.

A problem where the stability restriction is far more severe than the accuracy requirement is called **stiff**, and stiffness is ubiquitous: chemical kinetics with fast and slow reactions, circuits with widely separated time constants, spatially discretized diffusion equations, and multi-timescale biological models. The response — implicit methods, A-stability, and the nonlinear solve hiding in each step — organizes the second half of this topic.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Definition 1 (One-step method).** A one-step method is $\mathbf{y}_{n+1} = \mathbf{y}_n + h\,\Phi(t_n, \mathbf{y}_n, h)$ for an *increment function* $\Phi$. It is *explicit* if $\Phi$ does not depend on $\mathbf{y}_{n+1}$, and *implicit* otherwise.

**Definition 2 (Local truncation error).** Insert the exact solution into the method and measure the defect:

$$
\tau_{n+1}(h) = \frac{\mathbf{y}(t_{n+1}) - \mathbf{y}(t_n)}{h} - \Phi\bigl(t_n, \mathbf{y}(t_n), h\bigr) .
$$

The method is *consistent* if $\tau \to 0$ as $h \to 0$, and has **order $p$** if $\tau_{n+1}(h) = O(h^{p})$ uniformly, equivalently if one step starting from exact data is wrong by $O(h^{p+1})$.

**Definition 3 (Global error and convergence).** $\mathbf{e}_n = \mathbf{y}_n - \mathbf{y}(t_n)$. The method is *convergent of order $p$* if $\max_{0 \le n \le N}\Vert \mathbf{e}_n \Vert = O(h^{p})$ as $h \to 0$ with $Nh = T - t_0$ fixed.

**Definition 4 (Zero-stability).** A linear multistep method $\sum_{j=0}^{k}\alpha_j \mathbf{y}_{n+j} = h\sum_{j=0}^{k}\beta_j\mathbf{f}_{n+j}$ is *zero-stable* if the roots of its first characteristic polynomial $\rho(\zeta) = \sum_{j}\alpha_j\zeta^{j}$ satisfy $\vert\zeta\vert \le 1$, with any root on the unit circle simple (the *root condition*). All consistent one-step methods are automatically zero-stable.

**Definition 5 (Stability function and region of absolute stability).** Apply a method to the Dahlquist test equation $y' = \lambda y$, $\lambda \in \mathbb{C}$. Every one-step method reduces to $y_{n+1} = R(z)\,y_n$ with $z = h\lambda$; $R$ is the *stability function*. The *region of absolute stability* is

$$
\mathcal{S} = \{ z \in \mathbb{C} : \vert R(z) \vert \le 1 \} .
$$

**Definition 6 (A-stability, L-stability).** A method is **A-stable** if $\mathbb{C}^{-} = \{z : \operatorname{Re} z \lt 0\} \subseteq \mathcal{S}$, i.e. it is stable for every step size whenever the true solution decays. It is **L-stable** if in addition $\vert R(z)\vert \to 0$ as $\operatorname{Re} z \to -\infty$, so that fast transients are damped rather than merely bounded.

**Definition 7 (Stiffness ratio).** For a linear system $\mathbf{y}' = A\mathbf{y}$ with eigenvalues $\lambda_i$ all having negative real part, the *stiffness ratio* is $\max_i \vert\operatorname{Re}\lambda_i\vert / \min_i\vert\operatorname{Re}\lambda_i\vert$. A problem is *stiff* on $[t_0, T]$ when this ratio is large relative to the number of steps accuracy would demand.

**Definition 8 (Explicit Runge–Kutta method and Butcher tableau).** An $s$-stage explicit RK method is

$$
\mathbf{k}_i = \mathbf{f}\Bigl(t_n + c_i h,\ \mathbf{y}_n + h\sum_{j \lt i} a_{ij}\mathbf{k}_j\Bigr), \qquad \mathbf{y}_{n+1} = \mathbf{y}_n + h\sum_{i=1}^{s} b_i \mathbf{k}_i ,
$$

recorded as the tableau with nodes $\mathbf{c}$, matrix $A = (a_{ij})$ strictly lower triangular, and weights $\mathbf{b}$. Consistency requires $\sum_i b_i = 1$ and, in standard form, $c_i = \sum_j a_{ij}$.

**Definition 9 (Symplectic map).** A map $\Psi$ on phase space $(\mathbf{q}, \mathbf{p}) \in \mathbb{R}^{2d}$ is *symplectic* if its Jacobian $M = \partial\Psi/\partial(\mathbf{q},\mathbf{p})$ satisfies $M^{\top}J M = J$ with $J = \begin{bmatrix} 0 & I \\ -I & 0\end{bmatrix}$; equivalently it preserves the differential 2-form $d\mathbf{q}\wedge d\mathbf{p}$ and hence phase-space volume.

**Theorem 1 (Forward Euler: local and global error).** Let $\mathbf{f}$ be Lipschitz in $\mathbf{y}$ with constant $L$ and let $\Vert \mathbf{y}'' \Vert_\infty \le M$ on $[t_0, T]$. Then forward Euler has local truncation error $\vert\tau_n\vert \le \tfrac{M h}{2}$ and global error

$$
\Vert \mathbf{y}_n - \mathbf{y}(t_n) \Vert \le \frac{Mh}{2L}\left(e^{L(t_n - t_0)} - 1\right) + \Vert \mathbf{e}_0 \Vert e^{L(t_n - t_0)} = O(h) .
$$

**Theorem 2 (Discrete Gronwall lemma).** If $e_{n+1} \le (1 + hL)e_n + C$ with $e_0 \ge 0$, $h, L, C \ge 0$, then

$$
e_n \le e^{nhL}e_0 + \frac{C}{hL}\left(e^{nhL} - 1\right) .
$$

**Theorem 3 (Dahlquist equivalence theorem).** For a linear multistep method, *consistency* (order $p \ge 1$) together with *zero-stability* is **equivalent** to *convergence*; and a zero-stable method of order $p$ is convergent of order $p$. Neither hypothesis alone suffices.

**Theorem 4 (First Dahlquist barrier).** A zero-stable $k$-step linear multistep method has order at most $k + 1$ for odd $k$ and $k + 2$ for even $k$; an explicit one has order at most $k$.

**Theorem 5 (Second Dahlquist barrier).** No explicit linear multistep method is A-stable; no A-stable linear multistep method has order greater than $2$; and among order-2 A-stable methods the trapezoidal rule has the smallest error constant. Likewise, **no explicit Runge–Kutta method is A-stable**, since its stability function is a polynomial in $z$ and therefore unbounded.

**Theorem 6 (Stability functions of the basic methods).**

| Method | Stability function $R(z)$ | Real stability interval | A-stable? |
| :--- | :--- | :--- | :--- |
| Forward Euler | $1 + z$ | $(-2, 0)$ | No |
| Backward Euler | $(1 - z)^{-1}$ | $(-\infty, 0)$ | Yes (L-stable) |
| Trapezoidal / Crank–Nicolson | $\dfrac{1 + z/2}{1 - z/2}$ | $(-\infty, 0)$ | Yes (not L-stable) |
| Heun / RK2 | $1 + z + \tfrac{z^2}{2}$ | $(-2, 0)$ | No |
| Classical RK4 | $1 + z + \tfrac{z^2}{2} + \tfrac{z^3}{6} + \tfrac{z^4}{24}$ | $(-2.7853, 0)$ | No |

**Theorem 7 (RK4 order and cost).** The classical four-stage method with tableau nodes $\mathbf{c} = (0, \tfrac12, \tfrac12, 1)$ and weights $\mathbf{b} = (\tfrac16, \tfrac13, \tfrac13, \tfrac16)$ has order $4$, using $4$ evaluations of $\mathbf{f}$ per step. For $s \le 4$ an explicit $s$-stage method can attain order $s$; for order $5$ at least $6$ stages are required, and for order $8$ at least $11$ (the *Butcher barriers*).

**Theorem 8 (Symplecticity of symplectic Euler and Störmer–Verlet).** For a separable Hamiltonian $H(\mathbf{q},\mathbf{p}) = T(\mathbf{p}) + V(\mathbf{q})$, the maps

$$
\mathbf{p}_{n+1} = \mathbf{p}_n - h\nabla V(\mathbf{q}_n), \qquad \mathbf{q}_{n+1} = \mathbf{q}_n + h\nabla T(\mathbf{p}_{n+1})
$$

(symplectic Euler) and its symmetric composition (Störmer–Verlet, order 2) are symplectic. By backward error analysis they exactly conserve a *modified* Hamiltonian $\tilde{H} = H + O(h^{p})$, so the energy error stays bounded — it does **not** drift — over times exponentially long in $1/h$.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 1 — Forward Euler: local truncation error is $O(h^2)$ per step

**Claim.** If $\mathbf{y} \in C^{2}$, one step of forward Euler from exact data is wrong by $\tfrac{h^2}{2}\mathbf{y}''(\xi)$, i.e. $\tau_{n+1} = \tfrac{h}{2}\mathbf{y}''(\xi_n) = O(h)$, so the method has order $p = 1$.

**Derivation from three viewpoints.**

*Taylor.* Expand the exact solution about $t_n$ with Lagrange remainder:

$$
y(t_{n+1}) = y(t_n) + h y'(t_n) + \frac{h^2}{2}y''(\xi_n) = y(t_n) + h f(t_n, y(t_n)) + \frac{h^2}{2}y''(\xi_n),
$$

using the ODE to replace $y'$. Forward Euler drops the last term, so its one-step defect is exactly $\frac{h^2}{2}y''(\xi_n)$, and dividing by $h$ gives $\tau_{n+1} = \frac{h}{2}y''(\xi_n)$, bounded by $\frac{Mh}{2}$.

*Quadrature.* The exact update is $y(t_{n+1}) - y(t_n) = \int_{t_n}^{t_{n+1}}f(s, y(s))\,ds$, and Euler replaces the integrand by its left-endpoint value. The left-rectangle rule on an interval of width $h$ has error $\frac{h^2}{2}g'(\xi)$ for $g(s) = f(s, y(s))$, and $g' = y''$ — the same constant, arrived at through Topic 06.

*Geometric.* Euler follows the tangent line to the true trajectory. Over a step the trajectory departs from its tangent by $\frac{1}{2}\times(\text{curvature})\times h^2$, which is the same $\frac{h^2}{2}y''$. $\blacksquare$

**The consequence to remember.** A method whose *one-step* defect is $O(h^{p+1})$ is said to have order $p$, because the $1/h$ accumulation of steps removes exactly one power. Euler: defect $O(h^2)$, order $1$.

### Proof 2 — Forward Euler: global error is $O(h)$, via the discrete Gronwall lemma

**Claim.** (Theorems 1 and 2.) Under a Lipschitz condition $\Vert f(t,u) - f(t,v)\Vert \le L\Vert u - v\Vert$ and $\Vert y''\Vert_\infty \le M$,

$$
\Vert e_n \Vert \le \frac{Mh}{2L}\left(e^{L(t_n - t_0)} - 1\right) .
$$

**Proof.** *Step 1: one-step error recursion.* Subtract the exact relation $y(t_{n+1}) = y(t_n) + hf(t_n, y(t_n)) + \tfrac{h^2}{2}y''(\xi_n)$ from the scheme $y_{n+1} = y_n + hf(t_n, y_n)$:

$$
e_{n+1} = e_n + h\bigl[f(t_n, y_n) - f(t_n, y(t_n))\bigr] - \frac{h^2}{2}y''(\xi_n) .
$$

Take norms and apply the Lipschitz bound to the bracket and $M$ to the remainder:

$$
\Vert e_{n+1}\Vert \le \Vert e_n\Vert + hL\Vert e_n\Vert + \frac{Mh^2}{2} = (1 + hL)\Vert e_n \Vert + \frac{Mh^2}{2} .
$$

This is the crucial structural statement: **errors are amplified by $(1 + hL)$ each step and topped up by the local error.**

*Step 2: solve the recursion (discrete Gronwall).* Write $E_n = \Vert e_n\Vert$, $a = 1 + hL$, $C = Mh^2/2$. Unrolling $E_{n+1} \le aE_n + C$ gives the geometric sum

$$
E_n \le a^{n}E_0 + C\sum_{j=0}^{n-1}a^{j} = a^{n}E_0 + C\,\frac{a^{n} - 1}{a - 1} .
$$

*Step 3: bound $a^n$.* Since $1 + x \le e^{x}$ for all real $x$, $a^{n} = (1 + hL)^{n} \le e^{nhL} = e^{L(t_n - t_0)}$. With $a - 1 = hL$ and $E_0 = 0$ (exact initial data),

$$
\Vert e_n \Vert \le \frac{Mh^2/2}{hL}\left(e^{L(t_n - t_0)} - 1\right) = \frac{Mh}{2L}\left(e^{L(t_n - t_0)} - 1\right) = O(h). \qquad \blacksquare
$$

**Reading the bound.** The factor $h$ out front is the promised first-order convergence. The factor $e^{L(T - t_0)}$ is the honest price of a long integration of a system whose trajectories separate: on a chaotic problem with $L(T - t_0) = 30$, the amplification is $10^{13}$ and no step size saves you — which is precisely why weather forecasts have a horizon. The same argument applies verbatim to any consistent one-step method of order $p$ with $\Phi$ Lipschitz in $y$, giving global error $O(h^{p})$; that is the one-step case of Dahlquist's theorem.

### Proof 3 — Absolute stability of forward versus backward Euler

**Claim.** (Theorem 6.) On $y' = \lambda y$, forward Euler is stable iff $\vert 1 + h\lambda \vert \le 1$, giving the real interval $-2 \le h\lambda \le 0$; backward Euler is stable iff $\vert 1 - h\lambda\vert \ge 1$, which holds for every $h \gt 0$ when $\operatorname{Re}\lambda \lt 0$ — it is A-stable, and in fact L-stable.

**Proof.** *Forward Euler.* $y_{n+1} = y_n + h\lambda y_n = (1 + z)y_n$ with $z = h\lambda$, so $y_n = (1+z)^{n}y_0$ and $R(z) = 1 + z$. The numerical solution decays iff $\vert 1 + z\vert \le 1$: the closed **disc of radius 1 centred at $-1$**. For real $\lambda \lt 0$ this is $-2 \le h\lambda \le 0$, i.e.

$$
h \le \frac{2}{\vert\lambda\vert} .
$$

*Backward Euler.* $y_{n+1} = y_n + h\lambda y_{n+1} \Rightarrow y_{n+1} = \frac{1}{1 - z}y_n$, so $R(z) = (1-z)^{-1}$. Stability requires $\vert 1 - z\vert \ge 1$: the **exterior of the disc of radius 1 centred at $+1$**, which contains the entire left half-plane. Hence A-stability. Moreover $\vert R(z)\vert \to 0$ as $\operatorname{Re}z \to -\infty$, so stiff transients are *annihilated* rather than merely bounded — L-stability. $\blacksquare$

**Trapezoidal rule.** $y_{n+1} = y_n + \tfrac{h}{2}\lambda(y_n + y_{n+1})$ gives

$$
R(z) = \frac{1 + z/2}{1 - z/2}, \qquad \vert R(z)\vert \le 1 \iff \vert 1 + z/2\vert \le \vert 1 - z/2\vert \iff \operatorname{Re}z \le 0 ,
$$

because $\vert 1 + w\vert \le \vert 1 - w\vert$ says exactly that $w$ is at least as close to $-1$ as to $+1$, i.e. $\operatorname{Re}w \le 0$. So the trapezoidal rule is A-stable with stability region *exactly* the left half-plane — the best possible. But $R(z) \to -1$ as $\operatorname{Re}z \to -\infty$, so it is **not** L-stable: very stiff modes are not damped, they oscillate with sign flips at nearly constant magnitude. That ringing is the practical reason stiff production solvers prefer BDF2 or L-stable implicit RK over Crank–Nicolson.

**Numerical illustration** on $y' = -2y$, $y(0)=1$, $h = 0.1$, at $t = 0.5$ (exact $e^{-1} = 0.367879$): forward Euler gives $0.8^{5} = 0.327680$; backward Euler gives $1.2^{-5} = 0.401878$; the trapezoidal rule gives $(0.9/1.1)^{5} = 0.366648$ — second order and visibly closer, exactly as the orders predict.

### Proof 4 — Deriving the order conditions for a two-stage explicit Runge–Kutta method

**Claim.** The family

$$
k_1 = f(t_n, y_n), \quad k_2 = f(t_n + \alpha h,\ y_n + \alpha h k_1), \quad y_{n+1} = y_n + h(b_1 k_1 + b_2 k_2)
$$

has order $2$ if and only if $b_1 + b_2 = 1$ and $b_2\alpha = \tfrac12$; this gives Heun ($\alpha = 1$, $b = (\tfrac12,\tfrac12)$), the midpoint method ($\alpha = \tfrac12$, $b = (0,1)$), and Ralston ($\alpha = \tfrac23$, $b = (\tfrac14,\tfrac34)$).

**Proof.** *Exact side.* Using $y' = f$ and the chain rule, $y'' = f_t + f_y f$, so

$$
y(t_n + h) = y_n + hf + \frac{h^2}{2}(f_t + f_y f) + O(h^3),
$$

with everything evaluated at $(t_n, y_n)$.

*Method side.* Expand $k_2$ as a bivariate Taylor series about $(t_n, y_n)$:

$$
k_2 = f + \alpha h f_t + \alpha h f\,f_y + O(h^2) .
$$

Therefore

$$
y_{n+1} = y_n + h\bigl[(b_1 + b_2) f\bigr] + h^2\bigl[b_2\alpha (f_t + f_y f)\bigr] + O(h^3) .
$$

*Match.* Comparing the $O(h)$ terms forces $b_1 + b_2 = 1$ (consistency); comparing the $O(h^2)$ terms forces $b_2\alpha = \tfrac12$. Two equations, three unknowns: a one-parameter family, all of order exactly 2. $\blacksquare$

**Why order 3 is impossible here.** At $O(h^3)$ the exact solution produces the three independent elementary differentials $f_{tt} + 2ff_{ty} + f^2 f_{yy}$, $f_y(f_t + ff_y)$, weighted $\tfrac16$ and $\tfrac16$; a two-stage method can only generate the combination $\tfrac12 b_2\alpha^2(f_{tt} + 2ff_{ty} + f^2f_{yy})$ and *nothing* proportional to $f_y f_t$, so the two conditions $\tfrac12 b_2\alpha^2 = \tfrac16$ and $0 = \tfrac16$ are inconsistent. This is the smallest instance of Butcher's theory: order conditions are indexed by *rooted trees*, one condition per tree, and the number of trees ($1, 2, 4, 8, 17, \ldots$ cumulative for orders $1,2,3,4,5$) explains why RK4 needs $8$ conditions and why order $5$ needs $6$ stages rather than $5$.

**Classical RK4** satisfies all eight order-4 conditions with only four stages:

$$
\begin{array}{c|cccc} 0 & & & & \\ \tfrac12 & \tfrac12 & & & \\ \tfrac12 & 0 & \tfrac12 & & \\ 1 & 0 & 0 & 1 & \\ \hline & \tfrac16 & \tfrac13 & \tfrac13 & \tfrac16\end{array}
$$

Its weights $(\tfrac16, \tfrac13, \tfrac13, \tfrac16)$ are Simpson's rule, which is no accident: for $f$ independent of $y$ the method *is* Simpson's rule on $\int f\,dt$.

### Proof 5 — Consistency plus zero-stability implies convergence (the one-step case, and why the root condition is needed)

**Claim.** (Theorem 3, one-step case.) If a one-step method $y_{n+1} = y_n + h\Phi(t_n, y_n, h)$ has order $p$ (i.e. $\vert\tau_n\vert \le Ch^{p}$) and $\Phi$ is Lipschitz in $y$ with constant $L_\Phi$, then it converges with order $p$:

$$
\max_n \Vert e_n \Vert \le \frac{Ch^{p}}{L_\Phi}\left(e^{L_\Phi (T - t_0)} - 1\right) .
$$

**Proof.** Exactly the Gronwall argument of Proof 2 with the local error upgraded. By the definition of $\tau$, the exact solution satisfies $y(t_{n+1}) = y(t_n) + h\Phi(t_n, y(t_n), h) + h\tau_{n+1}$. Subtracting the scheme,

$$
\Vert e_{n+1}\Vert \le \Vert e_n\Vert + h L_\Phi\Vert e_n\Vert + h\vert\tau_{n+1}\vert \le (1 + hL_\Phi)\Vert e_n\Vert + Ch^{p+1},
$$

and the discrete Gronwall lemma (Theorem 2) with $C \to Ch^{p+1}$ gives the stated bound. $\blacksquare$

**Where zero-stability enters.** For one-step methods the amplification factor is automatically $1 + O(h)$, so zero-stability is free. For a $k$-step method it is not: the homogeneous recursion $\sum_j \alpha_j y_{n+j} = 0$ obtained at $h = 0$ has solutions $\zeta^{n}$ for each root $\zeta$ of $\rho(\zeta) = \sum_j\alpha_j\zeta^{j}$, and a root with $\vert\zeta\vert \gt 1$ makes rounding errors grow like $\vert\zeta\vert^{n}$ *independently of $h$* — no refinement helps.

**The canonical counterexample.** The two-step method

$$
y_{n+2} + 4y_{n+1} - 5y_n = h(4f_{n+1} + 2f_n)
$$

has order $3$ — the highest possible for a 2-step explicit method — yet $\rho(\zeta) = \zeta^2 + 4\zeta - 5 = (\zeta - 1)(\zeta + 5)$ has the root $\zeta = -5$. Applied to $y' = 0$ with a rounding perturbation of size $\delta$ in $y_1$, the numerical solution contains $\delta(-5)^{n}$ and explodes: at $n = 20$ the perturbation is amplified by $10^{14}$. High order without zero-stability is worthless — which is exactly the content of Dahlquist's equivalence theorem, and the reason the Adams and BDF families are constructed with $\rho$ having all its non-principal roots at zero.

### Proof 6 — Symplectic Euler preserves phase-space area, and why that matters more than order

**Claim.** (Theorem 8.) For the harmonic oscillator $H = \tfrac12(p^2 + q^2)$, the symplectic Euler map $p_{n+1} = p_n - hq_n$, $q_{n+1} = q_n + hp_{n+1}$ has Jacobian determinant exactly $1$, whereas forward Euler has determinant $1 + h^2 \gt 1$ (spiralling outward) and backward Euler has $1/(1+h^2) \lt 1$ (spiralling inward).

**Proof.** *Forward Euler:* $(q, p) \mapsto (q + hp,\ p - hq)$, with matrix

$$
M_{\text{FE}} = \begin{bmatrix} 1 & h \\ -h & 1\end{bmatrix}, \qquad \det M_{\text{FE}} = 1 + h^2 .
$$

Every step inflates phase-space area by the factor $1 + h^2$, so after $n$ steps the energy is multiplied by $(1+h^2)^{n} \approx e^{nh^2}$: the numerical oscillator gains energy without bound. Backward Euler is the inverse map, $\det = (1+h^2)^{-1}$, and loses energy the same way.

*Symplectic Euler:* substitute the first equation into the second, $q_{n+1} = q_n + h(p_n - hq_n) = (1 - h^2)q_n + hp_n$, so

$$
M_{\text{SE}} = \begin{bmatrix} 1 - h^2 & h \\ -h & 1\end{bmatrix}, \qquad \det M_{\text{SE}} = (1 - h^2) + h^2 = 1 .
$$

Exactly $1$, for every $h$ — the map is area preserving, i.e. symplectic in $d = 1$. $\blacksquare$

**Backward error analysis: the real reason it works.** Symplecticity alone would only bound the area. The stronger statement is that a symplectic method of order $p$ applied to a Hamiltonian system is the *exact* flow (up to exponentially small terms) of a nearby modified Hamiltonian $\tilde{H} = H + h^{p}H_p + \cdots$. Since the method conserves $\tilde{H}$ exactly and $\tilde{H} = H + O(h^{p})$, the true energy $H$ oscillates within an $O(h^{p})$ band **forever** rather than drifting linearly in $t$. For symplectic Euler on the harmonic oscillator one can check directly that the quadratic form $\tilde{H} = \tfrac12(p^2 + q^2 - hpq)$ is conserved exactly by $M_{\text{SE}}$.

**Consequence.** In a 10-million-step solar-system integration, RK4 (order 4, non-symplectic) shows a secular energy drift proportional to $t$, while Störmer–Verlet (order 2, symplectic) shows a bounded oscillation. Structure preservation beats order for long-time integration — which is why molecular dynamics, celestial mechanics, and Hamiltonian Monte Carlo all use Verlet-type leapfrog schemes.

## 4. Computational & Algorithmic Insights

### Adaptive step-size control and embedded pairs

Fixed steps are wasteful: they use the same $h$ in a boundary layer and on a flat plateau. Adaptive solvers estimate the local error each step and adjust. The trick is to obtain two approximations of different order from *one* set of stage evaluations — an **embedded pair**. Fehlberg's RKF45 uses six stages to produce order-4 and order-5 results; Dormand–Prince (`dopri5`, SciPy's `RK45`, MATLAB's `ode45`) uses seven with the FSAL ("first same as last") property, so it costs six new evaluations per step.

With $\hat{y}_{n+1}$ of order $p+1$ and $y_{n+1}$ of order $p$, the error estimate and the step update are

$$
\mathrm{err} = \Vert \hat{y}_{n+1} - y_{n+1} \Vert, \qquad h_{\text{new}} = h \left(\frac{\mathrm{tol}}{\mathrm{err}}\right)^{1/(p+1)} \cdot \theta ,
$$

with a safety factor $\theta \approx 0.8$–$0.9$ and clamps such as $0.2 \le h_{\text{new}}/h \le 5$. If $\mathrm{err} \gt \mathrm{tol}$ the step is *rejected* and retried with the smaller $h$. Example: $h = 0.1$, $\mathrm{err} = 10^{-4}$, $\mathrm{tol} = 10^{-6}$, $p = 4$, $\theta = 0.9$ gives $h_{\text{new}} = 0.9 \times 0.1 \times (10^{-2})^{1/5} \approx 0.0358$. Tolerances are normally mixed, $\mathrm{tol}_i = \mathrm{atol} + \mathrm{rtol}\cdot\vert y_i\vert$, so that components of very different magnitude are treated sensibly.

### Implicit solves, multistep methods, and choosing a solver

**The cost of being implicit.** Backward Euler requires solving $\mathbf{g}(\mathbf{y}_{n+1}) = \mathbf{y}_{n+1} - \mathbf{y}_n - h\mathbf{f}(t_{n+1}, \mathbf{y}_{n+1}) = \mathbf{0}$ at every step — a nonlinear system in $d$ unknowns (Topic 02). Fixed-point iteration converges only for $hL \lt 1$, which throws away the whole point on a stiff problem, so Newton's method is used with Jacobian $I - h\,\partial\mathbf{f}/\partial\mathbf{y}$. Production codes amortize this: they reuse a factorized Jacobian across steps ("modified Newton"), refactor only when convergence degrades, and exploit sparsity. The extra work per step is repaid many times over because $h$ can be $10^{3}$–$10^{6}$ times larger.

**Multistep methods** reuse already-computed $\mathbf{f}$ values instead of computing new stages. Adams–Bashforth (explicit) interpolates $\mathbf{f}$ through the last $k$ points and extrapolates; for example

$$
\text{AB2: } \mathbf{y}_{n+1} = \mathbf{y}_n + \frac{h}{2}\left(3\mathbf{f}_n - \mathbf{f}_{n-1}\right), \qquad \text{AM2 (trapezoid): } \mathbf{y}_{n+1} = \mathbf{y}_n + \frac{h}{2}\left(\mathbf{f}_{n+1} + \mathbf{f}_n\right) .
$$

Pairing them gives a **predictor–corrector** (PECE) scheme: one explicit step predicts, one implicit step corrects, at two evaluations per step regardless of order — far cheaper than RK of the same order. Their drawbacks are the need for a starting procedure, awkward step changes, and the Dahlquist barriers on stability. For stiff problems the **BDF** family (`ode15s`, SciPy's `BDF`) differentiates the interpolant rather than integrating it, and is stable up to order 6.

**Practical selection.**

| Situation | Method | Reason |
| :--- | :--- | :--- |
| Smooth, nonstiff, moderate accuracy | Dormand–Prince RK45 | Adaptive, 6 evaluations per step, robust default |
| Smooth, nonstiff, very high accuracy | Adams (`LSODA`) or extrapolation | High order at low cost per step |
| Stiff (kinetics, discretized diffusion) | BDF or Radau IIA | A- or L-stable, large steps permitted |
| Hamiltonian, long time | Störmer–Verlet, higher-order splitting | Bounded energy error, symplectic |
| Discontinuous right-hand side | Event detection + restart | Order collapses if a discontinuity is stepped over |

**Stiffness detection** in practice compares the step size a solver *wants* for accuracy with the one it is *forced* to take: if the ratio of rejected steps rises while the estimated error is tiny, the problem is stiff. `scipy.integrate.solve_ivp(..., method='LSODA')` automates the switch between an Adams solver and a BDF solver.

### The error floor and how to measure convergence

Total error is the sum of truncation and round-off contributions,

$$
E(h) \approx C_1 h^{p} + C_2\frac{\varepsilon_{\text{mach}}}{h},
$$

because each of the $T/h$ steps contributes about $\varepsilon_{\text{mach}}$ of rounding to the accumulated sum. Minimizing over $h$ gives $h_{\text{opt}} \sim (\varepsilon_{\text{mach}}/C)^{1/(p+1)}$: for RK4 in double precision, roughly $h \sim 10^{-3}$, below which further refinement makes things *worse*. High-order methods therefore reach the round-off floor at much larger (and far cheaper) step sizes — an underrated argument for order 5 or 8 over order 1.

**Verifying an implementation** is done by an order check: halve $h$ and measure the error ratio against a reference solution,

$$
p_{\text{observed}} = \log_2\frac{E(h)}{E(h/2)} .
$$

Forward Euler should give $\approx 1$, Heun $\approx 2$, RK4 $\approx 4$. A method that "works" but shows the wrong observed order almost always has a bug in a stage coefficient, and this test catches it immediately. On $y' = y - t^2 + 1$, $y(0) = 0.5$ with $h = 0.2$, one step gives errors $2.93\times 10^{-2}$ (Euler), $3.30\times 10^{-3}$ (Heun), $5.29\times 10^{-6}$ (RK4) — the local defects $O(h^{2})$, $O(h^{3})$, $O(h^{5})$ shrinking exactly as the orders predict.

## 5. Real-World Physics & AI/ML Applications

**Celestial mechanics.** The $N$-body problem is Hamiltonian and integrated for millions of orbits when studying solar-system stability or exoplanet resonances. Non-symplectic integrators produce a *secular* energy drift that grows linearly with $t$ and manufactures spurious orbital decay; symplectic Wisdom–Holman and higher-order Yoshida splittings keep the energy error bounded, which is why they, not RK4, are the standard.

**Molecular dynamics.** Velocity-Verlet integrates $10^{6}$–$10^{9}$ atoms for $10^{6}$ steps. The femtosecond step size is set by the fastest bond vibration — a stiffness constraint — and constraint algorithms (SHAKE/RATTLE) freeze those bonds precisely to relax it. Time reversibility and symplecticity ensure the simulation samples the correct statistical ensemble rather than heating or cooling artificially.

**Chemical kinetics and combustion.** Reaction networks routinely have rate constants spanning $10^{10}$, making them extremely stiff; explicit methods are unusable and BDF/Radau solvers with sparse Jacobians are mandatory. The Robertson problem is the standard three-species benchmark.

**Circuits, control, and PDEs.** SPICE integrates stiff circuit equations with trapezoidal and Gear (BDF) formulas. Method-of-lines discretization of the heat equation $u_t = u_{xx}$ produces $\mathbf{y}' = A\mathbf{y}$ with $\lambda_{\min} \approx -4/\Delta x^2$, so an explicit method needs $h \le \Delta x^2/2$ — the notorious parabolic stability restriction, which is why implicit Crank–Nicolson dominates diffusion solvers.

### Machine learning

- **Neural ODEs** (Chen et al., 2018) replace a discrete stack of residual blocks by a continuous flow $\dfrac{d\mathbf{h}}{dt} = f_\theta(\mathbf{h}, t)$, integrated by a standard solver from $t=0$ to $t=1$. Depth becomes integration time, memory becomes $O(1)$ in the number of "layers", and the solver's tolerance becomes an accuracy/compute dial adjustable at test time.
- **The adjoint sensitivity method** computes gradients without storing the forward trajectory: define $\mathbf{a}(t) = \partial L/\partial\mathbf{h}(t)$ and integrate the *backward* ODE $\dfrac{d\mathbf{a}}{dt} = -\mathbf{a}(t)^{\top}\dfrac{\partial f_\theta}{\partial \mathbf{h}}$ together with $\dfrac{dL}{d\theta} = -\int \mathbf{a}(t)^{\top}\dfrac{\partial f_\theta}{\partial\theta}\,dt$. This is continuous-time backpropagation, and it inherits every numerical concern of this topic — the backward solve can be stiff even when the forward one is not, and solver error becomes gradient error.
- **ResNets are forward Euler.** The residual update $\mathbf{h}_{k+1} = \mathbf{h}_k + f_\theta(\mathbf{h}_k)$ is exactly an Euler step with $h = 1$. Reading it that way explains why residual connections stabilize deep networks (they make the map $I + O(h)$ rather than a product of arbitrary matrices) and inspires architectures based on other integrators — midpoint/RK2 blocks, and reversible networks built from *symplectic* or leapfrog updates that allow activations to be recomputed rather than stored.
- **Diffusion models are ODE solvers.** Score-based generative models define a reverse-time SDE whose deterministic counterpart, the **probability-flow ODE**, produces the same marginals. Sampling is then literally numerical integration of $\dfrac{d\mathbf{x}}{dt} = f(\mathbf{x},t) - \tfrac12 g(t)^2\nabla_{\mathbf{x}}\log p_t(\mathbf{x})$, and "faster sampling" research is applied ODE numerics: DDIM is essentially an exponential-Euler step, DPM-Solver uses exponential integrators exploiting the semilinear structure, and Heun-style second-order samplers (EDM) cut the number of function evaluations from $1000$ to $\sim 20$. The number of denoising steps *is* the number of solver steps.
- **Gradient flow and optimization.** Gradient descent $\theta_{k+1} = \theta_k - \eta\nabla L(\theta_k)$ is forward Euler on the gradient flow $\dot{\theta} = -\nabla L(\theta)$, and the stability condition $\vert 1 - \eta\lambda\vert \le 1$ for the Hessian eigenvalues gives exactly the classical bound $\eta \lt 2/L_{\text{smooth}}$ — the learning-rate limit is an absolute-stability limit. Momentum corresponds to a second-order (heavy-ball) ODE, and Hamiltonian Monte Carlo requires a *symplectic* leapfrog integrator so that the Metropolis acceptance rate stays high.

### Summary of key results

| Result | Statement |
| :--- | :--- |
| Local vs global error | Local defect $O(h^{p+1})$ per step $\Rightarrow$ global error $O(h^{p})$ |
| Euler global bound | $\Vert e_n \Vert \le \frac{Mh}{2L}\bigl(e^{L(t_n-t_0)} - 1\bigr)$ |
| Dahlquist equivalence | consistency + zero-stability $\iff$ convergence |
| Stability function | $y_{n+1} = R(z)y_n$, $z = h\lambda$; stable iff $\vert R(z)\vert \le 1$ |
| Forward Euler | $R = 1+z$, real interval $(-2,0)$, so $h \le 2/\vert\lambda\vert$ |
| Backward Euler | $R = (1-z)^{-1}$, A-stable and L-stable |
| Trapezoidal | $R = \frac{1+z/2}{1-z/2}$, A-stable, order 2, not L-stable |
| RK4 | order 4, 4 stages, real stability interval $(-2.7853, 0)$ |
| Dahlquist barriers | no explicit LMM is A-stable; A-stable LMM order $\le 2$ |
| Step control | $h_{\text{new}} = \theta h(\mathrm{tol}/\mathrm{err})^{1/(p+1)}$ |
| Symplectic methods | conserve a modified $\tilde{H} = H + O(h^{p})$; energy error bounded, not drifting |

## 6. Canonical Literature Mapping & References

| Concept | Canonical source | Location |
| :--- | :--- | :--- |
| Euler, Taylor methods, error analysis | Burden & Faires, *Numerical Analysis* | Ch. 5.1–5.3 |
| Runge–Kutta methods and RKF45 | Burden & Faires | Ch. 5.4–5.5 |
| Multistep, predictor–corrector, stability | Burden & Faires | Ch. 5.6, 5.10–5.11 |
| Order conditions, rooted trees | Butcher, *Numerical Methods for ODEs* | Chs. 2–3 |
| Nonstiff theory, step control, dense output | Hairer, Nørsett & Wanner, *Solving ODEs I* | Chs. II.1–II.6 |
| Stiffness, A- and L-stability, BDF, Radau | Hairer & Wanner, *Solving ODEs II* | Chs. IV.1–IV.8 |
| Symplectic integrators, backward error analysis | Hairer, Lubich & Wanner, *Geometric Numerical Integration* | Chs. VI, IX |
| Zero-stability, Dahlquist equivalence and barriers | Iserles, *A First Course* | Chs. 2, 4 |
| Convergence and absolute stability, method of lines | LeVeque, *Finite Difference Methods* | Chs. 5–8 |
| Practical solver selection | Heath, *Scientific Computing* | Ch. 9 |
| Neural ODEs and adjoint sensitivity | Chen, Rubanova, Bettencourt & Duvenaud (2018) | NeurIPS |
| Diffusion samplers as ODE integrators | Song et al. (2021); Karras et al. (2022) | ICLR; NeurIPS |
| Embedded RK pairs (`dopri5`) | Dormand & Prince (1980) | JCAM 6(1) |

**Primary references.** Hairer, Nørsett & Wanner (Vol. I); Hairer & Wanner (Vol. II); Hairer, Lubich & Wanner (2006); Burden & Faires (Ch. 5); Butcher (2016); Iserles (Chs. 1–4); LeVeque (2007); Chen et al. (2018).

**Cross-links.** The analytical theory of ODEs — existence and uniqueness, phase-plane analysis, the matrix exponential, Laplace methods — lives in [`../../differential_equations/`](../../differential_equations/) and [`../../calculus/15_ordinary_differential_equations/`](../../calculus/15_ordinary_differential_equations/); the quadrature rules underlying every method are in [`../06_numerical_integration_quadrature/`](../06_numerical_integration_quadrature/); the Newton solves inside implicit steps are in [`../02_root_finding_methods/`](../02_root_finding_methods/).